In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 7: Uplift Modeling & CATE Estimation

### Business Context
A marketing campaign targets all users equally. Some users convert regardless (always-takers), some never convert (never-takers), and some are persuadables (respond to treatment). Targeting only persuadables reduces cost and maximizes incremental lift.

### Key Concepts

**Potential Outcomes Framework:**
```
ATE  = E[Y(1) - Y(0)]                  average treatment effect (all users)
ATT  = E[Y(1) - Y(0) | T=1]            effect on treated users
CATE = E[Y(1) - Y(0) | X=x]            individual-level effect given covariates
```

**Meta-Learners for CATE:**
- **T-Learner**: train separate outcome models μ_0(x), μ_1(x). CATE = μ_1(x) - μ_0(x).
- **S-Learner**: single model with treatment as a feature. CATE = f(x,1) - f(x,0).
- **X-Learner**: uses cross-predictions to impute counterfactuals. Better for imbalanced treatment/control.

**Qini Coefficient:** AUC of the uplift curve — measures how well the model identifies persuadables. A model with Qini=1 perfectly separates persuadables from always-takers and never-takers.

In [ ]:
# --- T-Learner and X-Learner for CATE ---

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

def generate_uplift_data(n=2000):
    X = np.random.randn(n, 5)
    T = (np.random.rand(n) < 0.5).astype(int)
    # True CATE depends on feature X[:,0]
    true_cate = 0.3 * X[:, 0] + 0.1
    y0 = 0.5 * X[:, 1] + np.random.randn(n) * 0.1    # control outcome
    y1 = y0 + true_cate + np.random.randn(n) * 0.1   # treated outcome
    y  = np.where(T == 1, y1, y0)
    return X, T, y, true_cate

class TLearner:
    def __init__(self):
        self.mu_0 = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.mu_1 = GradientBoostingRegressor(n_estimators=50, random_state=42)

    def fit(self, X, T, y):
        self.mu_0.fit(X[T == 0], y[T == 0])  # control model
        self.mu_1.fit(X[T == 1], y[T == 1])  # treatment model

    def predict_cate(self, X):
        return self.mu_1.predict(X) - self.mu_0.predict(X)

class XLearner:
    """X-Learner: better for imbalanced treatment/control sizes."""
    def __init__(self):
        self.mu_0   = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.mu_1   = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.tau_0  = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.tau_1  = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.propensity = GradientBoostingClassifier(n_estimators=50, random_state=42)

    def fit(self, X, T, y):
        X0, y0 = X[T == 0], y[T == 0]
        X1, y1 = X[T == 1], y[T == 1]

        self.mu_0.fit(X0, y0)
        self.mu_1.fit(X1, y1)

        # Impute counterfactuals and estimate individual treatment effects
        D0 = self.mu_1.predict(X0) - y0    # imputed effect for control units
        D1 = y1 - self.mu_0.predict(X1)    # imputed effect for treated units

        self.tau_0.fit(X0, D0)
        self.tau_1.fit(X1, D1)
        self.propensity.fit(X, T)

    def predict_cate(self, X):
        e = self.propensity.predict_proba(X)[:, 1]   # propensity score
        return e * self.tau_0.predict(X) + (1 - e) * self.tau_1.predict(X)

X, T, y, true_cate = generate_uplift_data(2000)
X_tr, X_te, T_tr, T_te, y_tr, y_te, tc_tr, tc_te = \
    train_test_split(X, T, y, true_cate, test_size=0.3, random_state=42)

t_learner = TLearner()
t_learner.fit(X_tr, T_tr, y_tr)

x_learner = XLearner()
x_learner.fit(X_tr, T_tr, y_tr)

t_preds = t_learner.predict_cate(X_te)
x_preds = x_learner.predict_cate(X_te)

t_mse = np.mean((t_preds - tc_te) ** 2)
x_mse = np.mean((x_preds - tc_te) ** 2)
print(f'CATE Estimation MSE:')
print(f'  T-Learner: {t_mse:.4f}')
print(f'  X-Learner: {x_mse:.4f}')
print(f'  True CATE range: [{tc_te.min():.2f}, {tc_te.max():.2f}]')

### L6 Interview Q&A

**Q: How is uplift modeling different from A/B testing?**  
A: A/B testing estimates the ATE (average treatment effect across all users). Uplift modeling estimates CATE (treatment effect per user). A/B tells you whether a campaign works on average. Uplift tells you WHO it works for, enabling targeted deployment to persuadables only — reducing cost and avoiding treatment of always-takers (who convert anyway) and never-takers (who don't respond).

**Q: Why is X-Learner better than T-Learner for imbalanced treatment/control?**  
A: T-Learner trains separate models on each arm. With a 10%/90% treatment/control split, the treatment model has very little data and generalizes poorly. X-Learner imputes counterfactuals using the better-trained model from the larger group, then trains the CATE models on the full dataset. The propensity-weighted combination further balances the estimates.

**Q: You deploy the uplift model and target only top-decile users. How do you measure success?**  
A: Measure incremental lift: compare conversion rate of targeted group vs. a holdout of similar users NOT targeted. The Qini coefficient measures ranking quality. Be careful — if you target fewer users, total conversions may drop even if per-user lift increases. Frame as cost-per-incremental-conversion vs. total reach trade-off.

### Common Pitfalls
- Confusing uplift with predicted conversion rate — high converters may not be persuadables
- Using A/B data from non-random assignment — selection bias invalidates CATE
- Not accounting for SUTVA (Stable Unit Treatment Value Assumption) — network effects